In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
%matplotlib inline 


def process_frame(image):
    
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) #11.40 blue, 29.99 red, 58.70 green
    kernel_size = 3
    blur_gray = cv2.GaussianBlur(gray, (kernel_size, kernel_size), 0)

    
    low_threshold = 100
    high_threshold = 150
    edges = cv2.Canny(blur_gray, low_threshold, high_threshold)

    mask = np.zeros_like(edges) #image with same height width or rows and colms    with zeros so it will be black 
    mask_color = 255 #mask color 255 means white 

    imshape = image.shape   # [Height, Width]
    vertices = np.array([
        [ (0, imshape[0]) , (200,int(imshape[0] / 1.7)) ,
         (int(imshape[1]-200), int(imshape[0] / 1.7))  ,  (imshape[1], imshape[0])] ], dtype=np.int32)

    mask=cv2.fillPoly(mask, vertices, mask_color)
    masked_edges = cv2.bitwise_and(edges, mask)
   

    # HOUGH HELPS TO FIND CERTAIN SHAPES OR LINES FROM THE IMAGE 
    rho = 5
    theta = np.pi / 180
    threshold = 100 #100 common lines
    min_line_length = 80
    max_line_gap = 200
    line_image = np.zeros_like(image) # 3 rgb channel blank or black image

    lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold, np.array([]),
                            min_line_length, max_line_gap)
    
    COLOR=[222,0,255]
    if lines is not None:
        for line in lines:
            
            for x1, y1, x2, y2 in line:
                # Calculate angle in degrees
                angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
                if 25<= angle <= 40:
                    cv2.line(line_image, (x1, y1), (x2, y2), COLOR, 8)

    # Overlay lines on the original image
    combo = cv2.addWeighted(image, 0.8, line_image, 1, 0)

    return combo
# Open the video file


In [4]:
cap = cv2.VideoCapture(r"/solidYellowLeft.mp4") #if you want to come out of the folder ../ if you want to enter folder in some folder 
# while in the same directory you can just write the name of the folder 

out = None
while cap.isOpened():     
    ret, frame = cap.read() # gives two values frame itself and indicator true or false  
    if not ret:   #not wll make True -- False and it converts False to True
        print("video breaked: or ended:")
        break
 
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    processed_frame = process_frame(frame_rgb)
    # a function inside a obj is known as method 
    
    processed_frame_bgr = cv2.cvtColor(processed_frame, cv2.COLOR_RGB2BGR)


    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # or use 'XVID', 'MJPG', etc. 
    if out is None:
        height, width = processed_frame_bgr.shape[:2]
        out = cv2.VideoWriter('new_video.mp4', fourcc, 25.0, (width, height))
    out.write(processed_frame_bgr)


    
    var1 = 100.0  # Target FPS just a float 
    wait_time = int(1000/ var1)   
    cv2.imshow('Lane Detection', processed_frame_bgr)
    if cv2.waitKey(wait_time) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()


AttributeError: 'NoneType' object has no attribute 'release'